# Base Model

## Imports

In [17]:
import random
import numpy as np
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Loading Data

In [18]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [19]:
print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

image, label = train_dataset[0]
print("\nImage shape:", image.shape)
print("Label:", label)

Training samples: 60000
Test samples: 10000

Image shape: torch.Size([1, 28, 28])
Label: 5


## Config Settings

In [20]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)

CUDA available: True
CUDA version: 12.1
Torch version: 2.5.1+cu121


In [21]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


## Model Code

### Model Addons

#### Mislabelling

In [22]:
def add_label_noise(dataset, noise_ratio=0.45, num_classes=10, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n = len(dataset.targets)
    n_noisy = int(noise_ratio * n)

    noisy_indices = torch.randperm(n)[:n_noisy]

    original = dataset.targets[noisy_indices]
    noise = torch.randint(0, num_classes - 1, size=(n_noisy,))
    noise = (noise + (noise >= original)).long()

    dataset.targets[noisy_indices] = noise

    return dataset

#### Mixup Augmentation

In [23]:
def mixup_data(x, y, alpha):
    lamb = np.random.beta(alpha, alpha)

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lamb * x + (1 - lamb) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lamb

In [24]:
def mixup_criterion(criterion, preds, y_a, y_b, lamb):
    return lamb * criterion(preds, y_a) + (1 - lamb) * criterion(preds, y_b)

#### Cutout Augmentation

In [25]:
class CutoutAugmentation():
    def __init__(self, K=16):
        self.K = K

    def __call__(self, img):
        # tensor shape: c, h, w
        if random.random() < 0.5:
            return img

        C, H, W = img.shape
        half = self.K // 2

        # random center
        cx = random.randint(0, W - 1)
        cy = random.randint(0, H - 1)

        # bounding box
        x1 = max(cx - half, 0)
        x2 = min(cx + half, W)
        y1 = max(cy - half, 0)
        y2 = min(cy + half, H)

        img[:, y1:y2, x1:x2] = 0.0
        return img

#### Standard Augmentation

In [26]:
class RandomShift():
    def __init__(self, K):
        self.K = K

    def __call__(self, img):
        k1 = random.randint(-self.K, self.K)
        k2 = random.randint(-self.K, self.K)

        C, H, W = img.shape
        shifted = torch.zeros_like(img)

        h_start = max(0, k1)
        h_end = min(H, H + k1)
        w_start = max(0, k2)
        w_end = min(W, W + k2)

        orig_h_start = max(0, -k1)
        orig_h_end = orig_h_start + (h_end - h_start)
        orig_w_start = max(0, -k2)
        orig_w_end = orig_w_start + (w_end - w_start)

        shifted[:, h_start:h_end, w_start:w_end] = img[:, orig_h_start:orig_h_end, orig_w_start:orig_w_end]

        img = shifted

        # horizontal flip
        # if random.random() < 0.5:
        #     img = torch.flip(img, dims=[2])

        return img

### Base Model Code

In [27]:
class ShallowNN(nn.Module):
    def __init__(self, hidden_size, p=0):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 10)
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [28]:
class Trainer:
    def __init__(self, hidden_size, learning_rate, epochs, batch_size, train_data, test_data, p=0):
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = ShallowNN(hidden_size=hidden_size, p=p).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        self.criterion = nn.CrossEntropyLoss()

        self.init_data_loaders(train_data, test_data)

    def init_data_loaders(self, train_data, test_data):
        self.train_loader = DataLoader(
            train_data,
            batch_size=self.batch_size,
            shuffle=True
        )

        self.test_loader = DataLoader(
            test_data,
            batch_size=1000,
            shuffle=False
        )

    def train(self, debug=False):
        train_accs = []
        test_accs = []
        losses = []

        for epoch in range(1, self.epochs + 1):
            self.model.train()

            correct = 0
            total = 0
            running_loss = 0.0

            for data, target in self.train_loader:
                data, target = data.to(self.device), target.to(self.device)

                self.optimizer.zero_grad()
                output = self.model(data)
                loss = self.criterion(output, target)
                loss.backward()
                self.optimizer.step()

                # loss
                running_loss += loss.item() * data.size(0)

                # train acc
                preds = output.argmax(dim=1)
                correct += (preds == target).sum().item()
                total += target.size(0)

            # metrics
            epoch_loss = running_loss / total
            train_acc = correct / total
            test_acc = self.evaluate()

            losses.append(epoch_loss)
            train_accs.append(train_acc)
            test_accs.append(test_acc)

            if debug:
                print(f"Epoch {epoch}/{self.epochs} | Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}, Loss: {epoch_loss:.4f}")
        
        return train_accs, test_accs, losses
    
    def mixup_train(self, alpha, debug=False):
        train_accs = []
        test_accs = []
        losses = []

        for epoch in range(1, self.epochs + 1):
            self.model.train()

            correct = 0.0
            total = 0
            running_loss = 0.0

            for data, target in self.train_loader:
                data, target = data.to(self.device), target.to(self.device)

                # mixup
                data, y_a, y_b, lamb = mixup_data(data, target, alpha)

                self.optimizer.zero_grad()
                output = self.model(data)

                loss = mixup_criterion(self.criterion, output, y_a, y_b, lamb)
                loss.backward()
                self.optimizer.step()

                # loss
                running_loss += loss.item() * data.size(0)

                # mixup train accuracy
                preds = output.argmax(dim=1)
                correct += (
                    lamb * (preds == y_a).sum().item()
                    + (1 - lamb) * (preds == y_b).sum().item()
                )
                total += target.size(0)

            # metrics
            epoch_loss = running_loss / total
            train_acc = correct / total
            test_acc = self.evaluate()

            losses.append(epoch_loss)
            train_accs.append(train_acc)
            test_accs.append(test_acc)

            if debug:
                print(f"Epoch {epoch}/{self.epochs} | Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}, Loss: {epoch_loss:.4f}")
        
        return train_accs, test_accs, losses

    def evaluate(self):
        self.model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in self.test_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                pred = output.argmax(dim=1)
                correct += (pred == target).sum().item()
                total += target.size(0)
        return correct / total

## Testing

### Base Model & Dropout

In [29]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=train_dataset,
    test_data=test_dataset,
    p=0
)

_, _, _ = trainer.train(debug=True)

Epoch 1/10 | Train Acc: 0.8526, Test Acc: 0.9155, Loss: 0.5451
Epoch 2/10 | Train Acc: 0.9244, Test Acc: 0.9345, Loss: 0.2626
Epoch 3/10 | Train Acc: 0.9396, Test Acc: 0.9453, Loss: 0.2090
Epoch 4/10 | Train Acc: 0.9499, Test Acc: 0.9530, Loss: 0.1734
Epoch 5/10 | Train Acc: 0.9571, Test Acc: 0.9588, Loss: 0.1474
Epoch 6/10 | Train Acc: 0.9629, Test Acc: 0.9606, Loss: 0.1280
Epoch 7/10 | Train Acc: 0.9666, Test Acc: 0.9644, Loss: 0.1138
Epoch 8/10 | Train Acc: 0.9711, Test Acc: 0.9647, Loss: 0.1005
Epoch 9/10 | Train Acc: 0.9730, Test Acc: 0.9673, Loss: 0.0924
Epoch 10/10 | Train Acc: 0.9755, Test Acc: 0.9673, Loss: 0.0837


In [30]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=train_dataset,
    test_data=test_dataset,
    p=0.4
)

_, _, _ = trainer.train(debug=True)

Epoch 1/10 | Train Acc: 0.7865, Test Acc: 0.9178, Loss: 0.7134
Epoch 2/10 | Train Acc: 0.8920, Test Acc: 0.9366, Loss: 0.3672
Epoch 3/10 | Train Acc: 0.9119, Test Acc: 0.9446, Loss: 0.3033
Epoch 4/10 | Train Acc: 0.9207, Test Acc: 0.9481, Loss: 0.2701
Epoch 5/10 | Train Acc: 0.9256, Test Acc: 0.9541, Loss: 0.2483
Epoch 6/10 | Train Acc: 0.9316, Test Acc: 0.9566, Loss: 0.2327
Epoch 7/10 | Train Acc: 0.9348, Test Acc: 0.9592, Loss: 0.2166
Epoch 8/10 | Train Acc: 0.9374, Test Acc: 0.9592, Loss: 0.2083
Epoch 9/10 | Train Acc: 0.9415, Test Acc: 0.9615, Loss: 0.1973
Epoch 10/10 | Train Acc: 0.9424, Test Acc: 0.9633, Loss: 0.1928


In [31]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=train_dataset,
    test_data=test_dataset,
    p=0.8
)

_, _, _ = trainer.train(debug=True)

Epoch 1/10 | Train Acc: 0.5499, Test Acc: 0.8960, Loss: 1.2756
Epoch 2/10 | Train Acc: 0.6767, Test Acc: 0.9108, Loss: 0.9122
Epoch 3/10 | Train Acc: 0.6980, Test Acc: 0.9184, Loss: 0.8464
Epoch 4/10 | Train Acc: 0.7160, Test Acc: 0.9207, Loss: 0.7988
Epoch 5/10 | Train Acc: 0.7213, Test Acc: 0.9228, Loss: 0.7777
Epoch 6/10 | Train Acc: 0.7284, Test Acc: 0.9259, Loss: 0.7571
Epoch 7/10 | Train Acc: 0.7338, Test Acc: 0.9266, Loss: 0.7432
Epoch 8/10 | Train Acc: 0.7341, Test Acc: 0.9288, Loss: 0.7356
Epoch 9/10 | Train Acc: 0.7399, Test Acc: 0.9279, Loss: 0.7270
Epoch 10/10 | Train Acc: 0.7421, Test Acc: 0.9297, Loss: 0.7173


### Noisy Labels

In [32]:
label_noise_ratio = 0.45

noisy_train_dataset = add_label_noise(copy.deepcopy(train_dataset), noise_ratio=label_noise_ratio)

In [33]:
# sanity testing to see noisy label thing worked
changed = (noisy_train_dataset.targets != train_dataset.targets).sum().item()
noise_fraction = changed / len(noisy_train_dataset)

print("Expected noise fraction:", label_noise_ratio)
print("Noise fraction:", noise_fraction)

Expected noise fraction: 0.45
Noise fraction: 0.45


In [34]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=noisy_train_dataset,
    test_data=test_dataset,
    p=0
)

_, _, _ = trainer.train(debug=True)

Epoch 1/10 | Train Acc: 0.4676, Test Acc: 0.9099, Loss: 1.8813
Epoch 2/10 | Train Acc: 0.5069, Test Acc: 0.9294, Loss: 1.7965
Epoch 3/10 | Train Acc: 0.5141, Test Acc: 0.9358, Loss: 1.7739
Epoch 4/10 | Train Acc: 0.5188, Test Acc: 0.9422, Loss: 1.7569
Epoch 5/10 | Train Acc: 0.5223, Test Acc: 0.9386, Loss: 1.7466
Epoch 6/10 | Train Acc: 0.5229, Test Acc: 0.9467, Loss: 1.7377
Epoch 7/10 | Train Acc: 0.5246, Test Acc: 0.9473, Loss: 1.7299
Epoch 8/10 | Train Acc: 0.5269, Test Acc: 0.9464, Loss: 1.7238
Epoch 9/10 | Train Acc: 0.5269, Test Acc: 0.9481, Loss: 1.7165
Epoch 10/10 | Train Acc: 0.5285, Test Acc: 0.9466, Loss: 1.7092


### Mixup Aug. Testing

In [35]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=train_dataset,
    test_data=test_dataset,
    p=0
)

_, _, _ = trainer.train(debug=True)

Epoch 1/10 | Train Acc: 0.8476, Test Acc: 0.9174, Loss: 0.5560
Epoch 2/10 | Train Acc: 0.9264, Test Acc: 0.9378, Loss: 0.2579
Epoch 3/10 | Train Acc: 0.9424, Test Acc: 0.9494, Loss: 0.2023
Epoch 4/10 | Train Acc: 0.9528, Test Acc: 0.9543, Loss: 0.1661
Epoch 5/10 | Train Acc: 0.9597, Test Acc: 0.9574, Loss: 0.1423
Epoch 6/10 | Train Acc: 0.9644, Test Acc: 0.9633, Loss: 0.1241
Epoch 7/10 | Train Acc: 0.9683, Test Acc: 0.9635, Loss: 0.1094
Epoch 8/10 | Train Acc: 0.9715, Test Acc: 0.9664, Loss: 0.0979
Epoch 9/10 | Train Acc: 0.9745, Test Acc: 0.9682, Loss: 0.0891
Epoch 10/10 | Train Acc: 0.9768, Test Acc: 0.9697, Loss: 0.0813


### Cutout Aug. Testing

In [36]:
cutout_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    CutoutAugmentation(K=16)
])

cutout_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=cutout_transform
)

In [37]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=cutout_train_dataset,
    test_data=test_dataset,
    p=0
)

_, _, _ = trainer.train(debug=True)

Epoch 1/10 | Train Acc: 0.7890, Test Acc: 0.9096, Loss: 0.7216
Epoch 2/10 | Train Acc: 0.8715, Test Acc: 0.9317, Loss: 0.4221
Epoch 3/10 | Train Acc: 0.8912, Test Acc: 0.9419, Loss: 0.3537
Epoch 4/10 | Train Acc: 0.9050, Test Acc: 0.9485, Loss: 0.3102
Epoch 5/10 | Train Acc: 0.9140, Test Acc: 0.9533, Loss: 0.2792
Epoch 6/10 | Train Acc: 0.9208, Test Acc: 0.9590, Loss: 0.2558
Epoch 7/10 | Train Acc: 0.9247, Test Acc: 0.9621, Loss: 0.2376
Epoch 8/10 | Train Acc: 0.9306, Test Acc: 0.9644, Loss: 0.2214
Epoch 9/10 | Train Acc: 0.9322, Test Acc: 0.9647, Loss: 0.2142
Epoch 10/10 | Train Acc: 0.9365, Test Acc: 0.9671, Loss: 0.2045


### Standard Aug. Testing

In [38]:
standard_aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    RandomShift(K=4)
])

standard_aug_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=standard_aug_transform
)

In [39]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=standard_aug_train_dataset,
    test_data=test_dataset,
    p=0
)

_, _, _ = trainer.train(debug=True)

Epoch 1/10 | Train Acc: 0.5941, Test Acc: 0.8246, Loss: 1.2668
Epoch 2/10 | Train Acc: 0.8133, Test Acc: 0.8836, Loss: 0.6401
Epoch 3/10 | Train Acc: 0.8647, Test Acc: 0.9063, Loss: 0.4683
Epoch 4/10 | Train Acc: 0.8845, Test Acc: 0.9209, Loss: 0.3927
Epoch 5/10 | Train Acc: 0.8992, Test Acc: 0.9234, Loss: 0.3480
Epoch 6/10 | Train Acc: 0.9075, Test Acc: 0.9291, Loss: 0.3190
Epoch 7/10 | Train Acc: 0.9142, Test Acc: 0.9402, Loss: 0.2958
Epoch 8/10 | Train Acc: 0.9181, Test Acc: 0.9404, Loss: 0.2787
Epoch 9/10 | Train Acc: 0.9215, Test Acc: 0.9444, Loss: 0.2660
Epoch 10/10 | Train Acc: 0.9237, Test Acc: 0.9450, Loss: 0.2575
